In [1]:
import torch
from torch import nn
from torch.optim import Adam
# load the model and dataset classes
from collections import defaultdict
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader,random_split
random.seed(42)

# data

In [2]:
test1 = torch.load("./../train_val_embedding_data/harmbench_actorattack.pt", map_location=torch.device('cpu'))
test2 = torch.load("./../train_val_embedding_data/harmbench_others.pt", map_location=torch.device('cpu'))

C:\Users\Karim Mahmoud\AppData\Local\Temp\ipykernel_28472\2319699421.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test1 = torch.load("./../train_val_embedding_data/ha

In [3]:
test=test1+test2

In [4]:
test[0][0].keys()

dict_keys(['u', 'y', 'score'])

# load ssm model

In [5]:
class stateSpaceModel(nn.Module):
    def __init__(self,state_dim, input_dim, hidden_dim1,hidden_dim2, output_dim):
        super(stateSpaceModel, self).__init__()
        # F(x_{t-1},u_t)=>x_t (Transition model)
        self.Fxu = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, state_dim)
        )

        # G(x_t,u_t)=>z_t (Observation model)
        self.Gxu = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, output_dim)
        )
        
    def forward(self, x_prev, u):
        ux_prev = torch.cat([u,x_prev], dim=-1)
        x_curr = self.Fxu(ux_prev)
        ux_curr = torch.cat([u,x_curr], dim=-1)
        zt = self.Gxu(ux_curr)
        return x_curr, zt

In [6]:
state_dim = 768    
input_dim = 768    
output_dim = 768   
hidden_dim_ssm1 = 1200  
hidden_dim_ssm2 = 900  

ssm=stateSpaceModel(state_dim=state_dim, input_dim=input_dim, hidden_dim1=hidden_dim_ssm1, hidden_dim2=hidden_dim_ssm2, output_dim=output_dim)

In [7]:
ssm.load_state_dict(torch.load("./../ssm_models/ssm_model_exp5/models_best_ssm.pth")['ssm']) 

C:\Users\Karim Mahmoud\AppData\Local\Temp\ipykernel_28472\4134113511.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ssm.load_state_dict(torch.load("./../ssm_models/ssm_

<All keys matched successfully>

# nbf

In [8]:
class NeuralBarrierFunction(nn.Module):
    """
    Neural Barrier Function (NBF) model.
    """
    def __init__(self, state_dim, input_dim, hidden_dim,class_num=5):
        super(NeuralBarrierFunction, self).__init__()
        self.nbf = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, class_num), 
        )

    def forward(self, x, u):
        xu = torch.cat([x, u], dim=-1)
        return self.nbf(xu)

In [9]:
hidden_dim_nbf = 32 
nbf=NeuralBarrierFunction(state_dim=state_dim, input_dim=input_dim, hidden_dim=hidden_dim_nbf)
nbf.load_state_dict(torch.load("./../paper_nbf_models/models_best_nbf_released.pth")['nbf'])

C:\Users\Karim Mahmoud\AppData\Local\Temp\ipykernel_28472\3664055741.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  nbf.load_state_dict(torch.load("./../paper_nbf_model

<All keys matched successfully>

In [10]:
predictions = []
scores = []

with torch.no_grad():
    nbf.eval()
    ssm.eval()
    for conv in test:
        x_prev=torch.zeros(state_dim)  
        for i, turn in enumerate(conv):
            u = turn['u']
            x_curr, zt = ssm(x_prev, u)
            x_prev = x_curr
            nbf_output = nbf(x_curr, u)
            nbf_output = torch.softmax(nbf_output, dim=-1).cpu().numpy()
            pred_unsafe=nbf_output[-1]
            pred_safe=max(nbf_output[:-1])
            h=pred_unsafe-pred_safe
            if h<0:
                predictions.append(0)  # Safe
                scores.append(turn["score"]==5)
            else:
                predictions.append(1)  # Unsafe
                scores.append(turn["score"]==5)
                break

In [11]:
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(scores, predictions))

              precision    recall  f1-score   support

       False       0.90      0.97      0.93      3768
        True       0.61      0.34      0.43       587

    accuracy                           0.88      4355
   macro avg       0.76      0.65      0.68      4355
weighted avg       0.86      0.88      0.87      4355



# test my model

In [12]:
# load the PCA model
import joblib
pca = joblib.load("pca_model.pkl")

d:\anaconda\envs\torch_gpu\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [13]:
# load prediction model
predictor=joblib.load("./mlruns/1/models/m-96f66a7f178a4a588f55e8c77e830c78/artifacts/model.pkl")

d:\anaconda\envs\torch_gpu\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [16]:
predictions = []
scores = []

with torch.no_grad():
    ssm.eval()
    for conv in test:
        x_prev=torch.zeros(state_dim)  
        for i, turn in enumerate(conv):
            u = turn['u']
            x_curr, zt = ssm(x_prev, u)
            x_prev = x_curr
            features=torch.concat([x_curr, turn["u"]], dim=0)
            features_pca = pca.transform(features.cpu().numpy().reshape(1, -1))
            pred=predictor.predict(features_pca)
            predictions.append(pred)
            scores.append(turn["score"]==5)

In [17]:
print(classification_report(scores, predictions))

              precision    recall  f1-score   support

       False       0.94      0.89      0.91      4116
        True       0.50      0.67      0.57       676

    accuracy                           0.86      4792
   macro avg       0.72      0.78      0.74      4792
weighted avg       0.88      0.86      0.87      4792



In [ ]:
model2=joblib.load("./../svm_model_v2_45mapping.pkl")

NameError: name 'joblib' is not defined

In [19]:
predictions = []
scores = []

with torch.no_grad():
    ssm.eval()
    for conv in test:
        x_prev=torch.zeros(state_dim)  
        for i, turn in enumerate(conv):
            u = turn['u']
            x_curr, zt = ssm(x_prev, u)
            x_prev = x_curr
            features=torch.concat([x_curr, turn["u"]], dim=0)
            features_pca = pca.transform(features.cpu().numpy().reshape(1, -1))
            pred=model2.predict(features_pca)
            predictions.append(pred)
            scores.append(turn["score"]==5 or turn["score"]==4)

In [20]:
print(classification_report(scores, predictions))

              precision    recall  f1-score   support

       False       0.84      0.83      0.83      3460
        True       0.57      0.59      0.58      1332

    accuracy                           0.76      4792
   macro avg       0.70      0.71      0.70      4792
weighted avg       0.76      0.76      0.76      4792

